In [6]:
# Cell 1 — настройки и импорт

from pathlib import Path
import re
from collections import defaultdict

SRC_DIR = Path(r"h:\Заморина ВСЕ чб jpg ОУДП")  # исходная папка
EXTS = {".jpg", ".jpeg", ".JPG", ".JPEG"}       # какие расширения считаем
SUFFIX_RE = re.compile(r"^(.*?)(_\d{2})$")      # ..._01, ..._02, ..._99

DRY_RUN = False  # безопасный прогон: ничего не создаём/не копируем


In [3]:
import os
import random
import shutil
from pathlib import Path

def rename_files_randomly(folder_path):
    """
    Переименовывает все файлы в папке в случайном порядке с порядковыми номерами
    """
    # Получаем абсолютный путь к папке
    folder_path = Path(folder_path).resolve()
    
    if not folder_path.exists() or not folder_path.is_dir():
        print(f"Ошибка: Папка '{folder_path}' не существует или не является директорией")
        return
    
    # Получаем список всех файлов в папке (исключаем папки)
    files = [f for f in folder_path.iterdir() if f.is_file()]
    
    if not files:
        print(f"В папке '{folder_path}' нет файлов")
        return
    
    print(f"Найдено {len(files)} файлов в папке: {folder_path}")
    
    # Перемешиваем файлы в случайном порядке
    random.shuffle(files)
    
    # Создаем временную папку для безопасного переименования
    temp_folder = folder_path / "_temp_rename"
    temp_folder.mkdir(exist_ok=True)
    
    renamed_files = []
    
    try:
        # Шаг 1: Копируем файлы во временную папку с новыми именами
        for i, file_path in enumerate(files, 1):
            # Получаем расширение файла
            extension = file_path.suffix
            
            # Получаем имя файла без расширения
            original_name = file_path.stem
            
            # Формируем новое имя с порядковым номером (формат: 001, 002, ...)
            new_name = f"{i:03d}_{original_name}{extension}"
            
            # Путь для временного файла
            temp_file_path = temp_folder / new_name
            
            # Копируем файл во временную папку с новым именем
            shutil.copy2(file_path, temp_file_path)
            
            renamed_files.append((file_path, temp_file_path, new_name))
            
            print(f"{i:03d}. {file_path.name} -> {new_name}")
        
        # Шаг 2: Удаляем оригинальные файлы
        for original_path, _, _ in renamed_files:
            original_path.unlink()
        
        # Шаг 3: Перемещаем переименованные файлы обратно в основную папку
        for _, temp_path, new_name in renamed_files:
            final_path = folder_path / new_name
            shutil.move(temp_path, final_path)
        
        # Удаляем временную папку
        temp_folder.rmdir()
        
        print(f"\n✓ Успешно переименовано {len(files)} файлов!")
        
    except Exception as e:
        print(f"\n✗ Произошла ошибка: {e}")
        print("Попытка восстановить оригинальные файлы...")
        
        # В случае ошибки пытаемся восстановить файлы
        try:
            # Удаляем временные файлы
            if temp_folder.exists():
                shutil.rmtree(temp_folder)
        except:
            pass
        
        print("Внимание: Некоторые файлы могли быть утеряны. Проверьте папку.")

def rename_files_in_place(folder_path):
    """
    Альтернативная версия: переименовывает файлы без временной папки
    (менее безопасно, но проще)
    """
    folder_path = Path(folder_path).resolve()
    
    if not folder_path.exists() or not folder_path.is_dir():
        print(f"Ошибка: Папка '{folder_path}' не существует или не является директорией")
        return
    
    files = [f for f in folder_path.iterdir() if f.is_file()]
    
    if not files:
        print(f"В папке '{folder_path}' нет файлов")
        return
    
    print(f"Найдено {len(files)} файлов")
    
    # Перемешиваем файлы
    random.shuffle(files)
    
    # Создаем список для временных имен
    temp_names = []
    
    # Первый проход: переименовываем во временные имена
    for i, file_path in enumerate(files, 1):
        extension = file_path.suffix
        original_name = file_path.stem
        
        # Создаем уникальное временное имя
        temp_name = f"temp_{i:03d}_{random.randint(1000, 9999)}{extension}"
        temp_path = folder_path / temp_name
        
        file_path.rename(temp_path)
        temp_names.append((temp_path, original_name, extension))
        
        print(f"Временное переименование: {file_path.name} -> {temp_name}")
    
    # Второй проход: переименовываем в финальные имена
    for i, (temp_path, original_name, extension) in enumerate(temp_names, 1):
        new_name = f"{i:03d}_{original_name}{extension}"
        final_path = folder_path / new_name
        
        temp_path.rename(final_path)
        print(f"Финальное переименование: {temp_path.name} -> {new_name}")
    
    print(f"\n✓ Успешно переименовано {len(files)} файлов!")

def main():
    # Укажите путь к вашей папке
    # Можно указать абсолютный путь или относительный
    folder_path = "."  # Текущая папка
    
    # Запрос пути у пользователя (если нужно)
    user_input = input(f"Введите путь к папке (по умолчанию: '{folder_path}'): ").strip()
    if user_input:
        folder_path = user_input
    
    print("\nВыберите метод переименования:")
    print("1. Безопасный метод (с временной папкой)")
    print("2. Простой метод (без временной папки)")
    
    choice = input("Введите номер метода (по умолчанию: 1): ").strip()
    
    if choice == "2":
        print("\nИспользуется простой метод...")
        rename_files_in_place(folder_path)
    else:
        print("\nИспользуется безопасный метод...")
        rename_files_randomly(folder_path)

if __name__ == "__main__":
    main()
# Использование:
#rename_with_subfolders("w:\\Projects\\Rear_ML\\Videos\\")

Введите путь к папке (по умолчанию: '.'):  w:\\Projects\\Rear_ML\\Videos\\



Выберите метод переименования:
1. Безопасный метод (с временной папкой)
2. Простой метод (без временной папки)


Введите номер метода (по умолчанию: 1):  2



Используется простой метод...
Найдено 76 файлов
Временное переименование: HOS_J05_2D.mp4 -> temp_001_3907.mp4
Временное переименование: LNOF_J05_4D.mp4 -> temp_002_7582.mp4
Временное переименование: NOF_H33_3D.mp4 -> temp_003_5983.mp4
Временное переименование: BOWL_F11_1D.mp4 -> temp_004_4625.mp4
Временное переименование: HOS_J05_5D.mp4 -> temp_005_6604.mp4
Временное переименование: NOF_H09_1D.mp4 -> temp_006_5475.mp4
Временное переименование: LNOF_J05_1D.mp4 -> temp_007_8867.mp4
Временное переименование: BOWL_F11_5D.mp4 -> temp_008_8288.mp4
Временное переименование: LNOF_J06_2D.mp4 -> temp_009_1589.mp4
Временное переименование: LNOF_J52_4D.mp4 -> temp_010_2126.mp4
Временное переименование: BOWL_F11_4D.mp4 -> temp_011_9228.mp4
Временное переименование: BOWL_F11_3D.mp4 -> temp_012_3643.mp4
Временное переименование: BOWL_J19_2D.mp4 -> temp_013_3471.mp4
Временное переименование: LNOF_J24_2D.mp4 -> temp_014_2489.mp4
Временное переименование: BOWL_F07_3D.mp4 -> temp_015_3702.mp4
Временное 

In [4]:
# Cell 2 — сбор файлов и построение плана (без действий на диске)

def build_plan(src_dir: Path):
    files = sorted([p for p in src_dir.iterdir() if p.is_file() and p.suffix in EXTS])

    groups = defaultdict(list)
    skipped = []

    for p in files:
        stem = p.stem  # имя без расширения
        m = SUFFIX_RE.match(stem)
        if not m:
            skipped.append(p)
            continue

        prefix, suffix = m.group(1), m.group(2)
        # папка = prefix, новое имя = suffix + исходное расширение (сохраняем .jpg/.jpeg как есть)
        dst_dir = src_dir / prefix
        dst_name = f"{suffix}{p.suffix}"
        dst_path = dst_dir / dst_name

        groups[prefix].append((p, dst_path))

    return groups, skipped


groups, skipped = build_plan(SRC_DIR)

print(f"Найдено файлов подходящих под шаблон *_NN.jpg: {sum(len(v) for v in groups.values())}")
print(f"Будет создано (теоретически) папок: {len(groups)}")
print(f"Пропущено (не совпало с *_NN): {len(skipped)}")
if skipped:
    print("\nПримеры пропущенных (первые 20):")
    for p in skipped[:20]:
        print("  -", p.name)


Найдено файлов подходящих под шаблон *_NN.jpg: 4851
Будет создано (теоретически) папок: 352
Пропущено (не совпало с *_NN): 0


In [5]:
# Cell 3 — вывод подробного плана и проверка коллизий имён (ещё без действий)

def show_plan(groups, max_groups=10, max_files_per_group=20):
    prefixes = sorted(groups.keys())
    print("ПЛАН (dry-run):\n")

    for i, prefix in enumerate(prefixes[:max_groups], 1):
        items = groups[prefix]
        print(f"[{i}] Папка: {prefix}  | файлов: {len(items)}")
        for src, dst in items[:max_files_per_group]:
            print(f"    COPY: {src.name}  ->  {dst.relative_to(SRC_DIR)}")
        if len(items) > max_files_per_group:
            print(f"    ... ещё {len(items) - max_files_per_group} файлов")
        print()

    if len(prefixes) > max_groups:
        print(f"... ещё {len(prefixes) - max_groups} папок (не показаны)\n")

def check_collisions(groups):
    # коллизии: два разных исходника дают один и тот же путь назначения внутри одной папки
    collisions = []
    for prefix, items in groups.items():
        seen = {}
        for src, dst in items:
            if dst in seen:
                collisions.append((prefix, seen[dst], src, dst))
            else:
                seen[dst] = src
    return collisions

show_plan(groups)

collisions = check_collisions(groups)
print("Коллизии имён назначения:", len(collisions))
if collisions:
    print("\nПервые 10 коллизий:")
    for prefix, src1, src2, dst in collisions[:10]:
        print(f"  [{prefix}] {src1.name} и {src2.name} -> {dst.name}")


ПЛАН (dry-run):

[1] Папка: 10153G_3m_FC  | файлов: 15
    COPY: 10153G_3m_FC_01.jpg  ->  10153G_3m_FC\_01.jpg
    COPY: 10153G_3m_FC_02.jpg  ->  10153G_3m_FC\_02.jpg
    COPY: 10153G_3m_FC_03.jpg  ->  10153G_3m_FC\_03.jpg
    COPY: 10153G_3m_FC_04.jpg  ->  10153G_3m_FC\_04.jpg
    COPY: 10153G_3m_FC_05.jpg  ->  10153G_3m_FC\_05.jpg
    COPY: 10153G_3m_FC_06.jpg  ->  10153G_3m_FC\_06.jpg
    COPY: 10153G_3m_FC_07.jpg  ->  10153G_3m_FC\_07.jpg
    COPY: 10153G_3m_FC_08.jpg  ->  10153G_3m_FC\_08.jpg
    COPY: 10153G_3m_FC_09.jpg  ->  10153G_3m_FC\_09.jpg
    COPY: 10153G_3m_FC_10.jpg  ->  10153G_3m_FC\_10.jpg
    COPY: 10153G_3m_FC_11.jpg  ->  10153G_3m_FC\_11.jpg
    COPY: 10153G_3m_FC_12.jpg  ->  10153G_3m_FC\_12.jpg
    COPY: 10153G_3m_FC_13.jpg  ->  10153G_3m_FC\_13.jpg
    COPY: 10153G_3m_FC_14.jpg  ->  10153G_3m_FC\_14.jpg
    COPY: 10153G_3m_FC_15.jpg  ->  10153G_3m_FC\_15.jpg

[2] Папка: 10153I_3m_AC  | файлов: 15
    COPY: 10153I_3m_AC_01.jpg  ->  10153I_3m_AC\_01.jpg
    COPY: 

In [7]:
# Cell 4 — реальное копирование (создаёт папки, копирует, логирует)
import shutil

def execute_plan(groups, dry_run=True):
    copied = 0
    created_dirs = 0
    already_exist = 0

    for prefix, items in sorted(groups.items()):
        dst_dir = SRC_DIR / prefix
        if not dst_dir.exists():
            if dry_run:
                print(f"MKDIR (dry-run): {dst_dir}")
            else:
                dst_dir.mkdir(parents=True, exist_ok=True)
                created_dirs += 1

        for src, dst in items:
            if dst.exists():
                already_exist += 1
                print(f"SKIP exists: {dst.relative_to(SRC_DIR)} (источник: {src.name})")
                continue

            if dry_run:
                print(f"COPY (dry-run): {src.name} -> {dst.relative_to(SRC_DIR)}")
            else:
                shutil.copy2(src, dst)
                copied += 1
                print(f"COPIED: {src.name} -> {dst.relative_to(SRC_DIR)}")

    print("\nИТОГО:")
    print("  создано папок:", created_dirs if not dry_run else "(dry-run не создаёт)")
    print("  скопировано файлов:", copied if not dry_run else "(dry-run не копирует)")
    print("  пропущено из-за существования назначения:", already_exist)

execute_plan(groups, dry_run=DRY_RUN)


COPIED: 10153G_3m_FC_01.jpg -> 10153G_3m_FC\_01.jpg
COPIED: 10153G_3m_FC_02.jpg -> 10153G_3m_FC\_02.jpg
COPIED: 10153G_3m_FC_03.jpg -> 10153G_3m_FC\_03.jpg
COPIED: 10153G_3m_FC_04.jpg -> 10153G_3m_FC\_04.jpg
COPIED: 10153G_3m_FC_05.jpg -> 10153G_3m_FC\_05.jpg
COPIED: 10153G_3m_FC_06.jpg -> 10153G_3m_FC\_06.jpg
COPIED: 10153G_3m_FC_07.jpg -> 10153G_3m_FC\_07.jpg
COPIED: 10153G_3m_FC_08.jpg -> 10153G_3m_FC\_08.jpg
COPIED: 10153G_3m_FC_09.jpg -> 10153G_3m_FC\_09.jpg
COPIED: 10153G_3m_FC_10.jpg -> 10153G_3m_FC\_10.jpg
COPIED: 10153G_3m_FC_11.jpg -> 10153G_3m_FC\_11.jpg
COPIED: 10153G_3m_FC_12.jpg -> 10153G_3m_FC\_12.jpg
COPIED: 10153G_3m_FC_13.jpg -> 10153G_3m_FC\_13.jpg
COPIED: 10153G_3m_FC_14.jpg -> 10153G_3m_FC\_14.jpg
COPIED: 10153G_3m_FC_15.jpg -> 10153G_3m_FC\_15.jpg
COPIED: 10153I_3m_AC_01.jpg -> 10153I_3m_AC\_01.jpg
COPIED: 10153I_3m_AC_02.jpg -> 10153I_3m_AC\_02.jpg
COPIED: 10153I_3m_AC_03.jpg -> 10153I_3m_AC\_03.jpg
COPIED: 10153I_3m_AC_04.jpg -> 10153I_3m_AC\_04.jpg
COPIED: 1015

In [6]:
import os

def rename_files_in_folders(parent_dir):
    for root_dir, dirnames, filenames in os.walk(parent_dir):
        for filename in filenames:
            if "_camera3" in filename:
                old_path = os.path.join(root_dir, filename)
                new_filename = filename.replace("_camera3", "_")
                new_path = os.path.join(root_dir, new_filename)
                os.rename(old_path, new_path)
                print(f"Переименовано: {old_path} -> {new_path}")

# Укажите название родительской папки, содержащей все нужные папки
parent_directory = "h:\\BOWL\\BehaviorSide2Data\\1_Raw2\\"
rename_files_in_folders(parent_directory)


Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera3.csv -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_.csv
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera30.avi -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_0.avi
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera31.avi -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_1.avi
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera310.avi -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_10.avi
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera311.avi -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_11.avi
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera312.avi -> h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_12.avi
Переименовано: h:\BOWL\BehaviorSide2Data\1_Raw2\BOF_F05_5D\BOF_F05_5D_camera313.avi -> h:\BOWL\BehaviorSide2Data\1

In [25]:
import os

def add_suffix_to_files(parent_dir, suffix):
    for root_dir, dirnames, filenames in os.walk(parent_dir):
        for filename in filenames:
            old_path = os.path.join(root_dir, filename)
            if len(filename) > 7:
                new_filename = filename[:7] + suffix + filename[7:]
                new_path = os.path.join(root_dir, new_filename)
                os.rename(old_path, new_path)
                print(f"Переименовано: {old_path} -> {new_path}")

# Укажите название родительской папки, содержащей все нужные папки
parent_directory = "h:\\M_mice\\CC\\MiniscopeData\\RawData\\"

# Добавление "_1D" после седьмого символа в каждом файле
add_suffix_to_files(parent_directory, "_1D")

Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_00.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D00.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_01.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D01.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_02.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D02.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_03.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D03.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_04.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D04.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_05.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D05.avi
Переименовано: h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02_06.avi -> h:\M_mice\CC\MiniscopeData\RawData\CC_M02_1D\CC_M02__1D06.avi
Переименовано: h:\M_mice\CC

In [2]:
% Укажите путь к папке с файлами
folderPath = 'i:\\_STFP\\_VideoData\\2_DLC\\test_dlc\\';

% Получаем список всех CSV файлов в папке
files = dir(fullfile(folderPath, '*.csv'));

% Обходим каждый файл и переименовываем его по заданному шаблону
for i = 1:length(files)
    % Исходное имя файла
    oldName = files(i).name;
    
    % Разбиваем имя файла на части по символу '_'
    parts = split(oldName, '_');
    
    % Определяем новый формат имени файла
    % STFP1_D1_DLC_resnet152_MiceUniversal_combined.csv должно стать STFP_A01_D1_T1_track.csv
    % STFP4_D2_T2_DLC_resnet152_MiceUniversal_combined.csv должно стать STFP_A04_D2_T2_track.csv
    
    % Получаем значение после 'STFP' и преобразуем его в нужный формат
    stfpNumber = parts{1}(5:end);  % '1' из 'STFP1'
    newStfpNumber = sprintf('A%02d', str2double(stfpNumber));  % Преобразуем '1' в 'A01'
    
    % Получаем и преобразуем другие части имени файла
    dayPart = parts{2};  % 'D1'
    if length(parts) > 3 && startsWith(parts{3}, 'T')
        timePart = parts{3};  % 'T2'
    else
        timePart = 'T1';  % Если время не указано, добавляем 'T1'
    end
    
    % Определяем суффикс для нового имени файла
    if contains(oldName, 'combined')
        suffix = 'track';
    else
        suffix = 'spikes';
    end
    
    % Формируем новое имя файла
    newName = sprintf('STFP_%s_%s_%s_%s.csv', newStfpNumber, dayPart, timePart, suffix);
    
    % Полный путь к старому и новому файлу
    oldFilePath = fullfile(folderPath, oldName);
    newFilePath = fullfile(folderPath, newName);
    
    % Переименовываем файл
    movefile(oldFilePath, newFilePath);
    
    % Выводим сообщение о переименовании
    disp(['Файл ' oldName ' переименован в ' newName]);
end


SyntaxError: invalid syntax (655419320.py, line 8)

## Удалить дату и время в имени файлов

In [7]:
import os
import re

# Укажите путь к папке с файлами
folder_path = 'w:\\Projects\\RFC\\6_Traces\\'

# Регулярное выражение для поиска части с датой и временем и префикса "1_Raw"
date_pattern = re.compile(r'(_\d{2}-\d{2}-\d{4} \d{2}-\d{2}-\d{2})')
#prefix_pattern = re.compile(r'^RFC_')

def rename_files_in_folder(folder_path):
    # Перебор всех файлов в папке
    for filename in os.listdir(folder_path):
        # Удаляем префикс "1_Raw_" в начале имени файла
        #new_filename = re.sub(prefix_pattern, '', filename)

        # Ищем и удаляем часть имени файла, связанную с датой и временем
        new_filename = re.sub(date_pattern, '', filename)
        print(new_filename)
        # Если новое имя отличается от старого, переименовываем файл
        if new_filename != filename:
            old_file_path = os.path.join(folder_path, filename)
            new_file_path = os.path.join(folder_path, new_filename)

            # Переименовываем файл
            os.rename(old_file_path, new_file_path)
            print(f"Переименован: {filename} -> {new_filename}")

# Вызов функции переименования
rename_files_in_folder(folder_path)



RFC_F01_1D_traces.csv
RFC_F01_3D_traces.csv
Переименован: RFC_F01_3D_08-11-2024 14-21-16_traces.csv -> RFC_F01_3D_traces.csv
RFC_F04_1D_traces.csv
Переименован: RFC_F04_1D_23-10-2024 17-41-29_traces.csv -> RFC_F04_1D_traces.csv
RFC_F04_3D_traces.csv
Переименован: RFC_F04_3D_13-11-2024 20-14-55_traces.csv -> RFC_F04_3D_traces.csv
RFC_F05_1D_traces.csv
RFC_F05_3D_traces.csv
Переименован: RFC_F05_3D_13-11-2024 20-29-06_traces.csv -> RFC_F05_3D_traces.csv
RFC_F06_1D_traces.csv
Переименован: RFC_F06_1D_28-10-2024 03-04-07_traces.csv -> RFC_F06_1D_traces.csv
RFC_F06_3D_traces.csv
Переименован: RFC_F06_3D_13-11-2024 20-55-30_traces.csv -> RFC_F06_3D_traces.csv
RFC_F07_1D_traces.csv
RFC_F07_3D_traces.csv
Переименован: RFC_F07_3D_13-11-2024 21-26-30_traces.csv -> RFC_F07_3D_traces.csv
RFC_F08_1D_traces.csv
Переименован: RFC_F08_1D_28-10-2024 03-25-52_traces.csv -> RFC_F08_1D_traces.csv
RFC_F08_3D_traces.csv
Переименован: RFC_F08_3D_14-11-2024 19-13-27_traces.csv -> RFC_F08_3D_traces.csv
RFC_F09

### Переименовать все папки и файлы внутри папки определенной, в именах нужно заменить 2D на 1D и любое xD, где x - число на 1

In [2]:
import os
import re

def rename_files_and_folders(root_folder):
    # Обход папок и файлов с использованием os.walk
    for dirpath, dirnames, filenames in os.walk(root_folder, topdown=False):
        # Переименование файлов
        for filename in filenames:
            old_path = os.path.join(dirpath, filename)
            new_filename = re.sub(r'\dD', '2D', filename)
            new_path = os.path.join(dirpath, new_filename)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименован файл: {old_path} -> {new_path}")
        
        # Переименование папок
        for dirname in dirnames:
            old_path = os.path.join(dirpath, dirname)
            new_dirname = re.sub(r'\dD', '2D', dirname)
            new_path = os.path.join(dirpath, new_dirname)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименована папка: {old_path} -> {new_path}")

# Укажите путь к корневой папке
root_folder = "c:\\Users\\admin\\Projects\\MSS\\MiniscopeData\\1_Raw\\Massed\\2D\\"
rename_files_and_folders(root_folder)


Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_5D_1T_0.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_2D_1T_0.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_5D_1T_1.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_2D_1T_1.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_5D_1T_10.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_2D_1T_10.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_5D_1T_11.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_2D_1T_11.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D\MSS_F07_5D_1T\MSS_F07_5D_1T_12.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed\2D

### всем файлам и папкам в папке после буквы D добавит _1T

In [ ]:
import os
import re

def add_suffix_after_d(root_folder):
    # Обход папок и файлов с использованием os.walk
    for dirpath, dirnames, filenames in os.walk(root_folder, topdown=False):
        # Переименование файлов
        for filename in filenames:
            old_path = os.path.join(dirpath, filename)
            new_filename = re.sub(r'(D)(?!.*_1T)', r'\1_1T', filename)
            new_path = os.path.join(dirpath, new_filename)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименован файл: {old_path} -> {new_path}")
        
        # Переименование папок
        for dirname in dirnames:
            old_path = os.path.join(dirpath, dirname)
            new_dirname = re.sub(r'(D)(?!.*_1T)', r'\1_1T', dirname)
            new_path = os.path.join(dirpath, new_dirname)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименована папка: {old_path} -> {new_path}")

# Укажите путь к корневой папке
root_folder = "c:\\Users\\admin\\Projects\\MSS\\MiniscopeData\\1_Raw\\sp12\\"
add_suffix_after_d(root_folder)


### заменить 6D на 2D_1T

In [1]:
import os
import re

def replace_6d_with_2d_1t(root_folder):
    # Обход папок и файлов с использованием os.walk
    for dirpath, dirnames, filenames in os.walk(root_folder, topdown=False):
        # Переименование файлов
        for filename in filenames:
            old_path = os.path.join(dirpath, filename)
            new_filename = filename.replace("6D", "2D_1T")  # Заменяем 6D на 2D_1T
            new_path = os.path.join(dirpath, new_filename)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименован файл: {old_path} -> {new_path}")
        
        # Переименование папок
        for dirname in dirnames:
            old_path = os.path.join(dirpath, dirname)
            new_dirname = dirname.replace("6D", "2D_1T")  # Заменяем 6D на 2D_1T
            new_path = os.path.join(dirpath, new_dirname)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименована папка: {old_path} -> {new_path}")

# Укажите путь к корневой папке
root_folder = "c:\\Users\\admin\\Projects\\MSS\\MiniscopeData\\1_Raw\\Massed_2wave\\2D\\"
replace_6d_with_2d_1t(root_folder)


Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_6D_00.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_2D_1T_00.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_6D_01.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_2D_1T_01.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_6D_02.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_2D_1T_02.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_6D_03.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_2D_1T_03.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\2D\MSS_H26_6D\MSS_H26_6D_04.avi -> c:\Users\admin\Projects\MSS\MiniscopeData

### заменить 5D1 на 1D_1T, 5D2 на 1D_2T и тд

In [3]:
import os
import re

def replace_5d_with_1d_xt(root_folder):
    # Обход папок и файлов с использованием os.walk
    for dirpath, dirnames, filenames in os.walk(root_folder, topdown=False):
        # Переименование файлов
        for filename in filenames:
            old_path = os.path.join(dirpath, filename)
            # Используем регулярное выражение для замены
            new_filename = re.sub(r'5D(\d)', r'1D_\1T', filename)
            new_path = os.path.join(dirpath, new_filename)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименован файл: {old_path} -> {new_path}")
        
        # Переименование папок
        for dirname in dirnames:
            old_path = os.path.join(dirpath, dirname)
            # Используем регулярное выражение для замены
            new_dirname = re.sub(r'5D(\d)', r'1D_\1T', dirname)
            new_path = os.path.join(dirpath, new_dirname)
            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Переименована папка: {old_path} -> {new_path}")

# Укажите путь к корневой папке
root_folder = "c:\\Users\\admin\\Projects\\MSS\\MiniscopeData\\1_Raw\\Massed_2wave\\1D\\"
replace_5d_with_1d_xt(root_folder)


Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_5D1_0.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_1D_1T_0.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_5D1_1.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_1D_1T_1.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_5D1_2.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_1D_1T_2.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_5D1_3.avi -> c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_1D_1T_3.avi
Переименован файл: c:\Users\admin\Projects\MSS\MiniscopeData\1_Raw\Massed_2wave\1D\MSS_H26_5D1\MSS_H26_5D1_Mini_TS.csv -> c:\Users\admin\Projects\MSS\Mi

### Парсинг таблиц boris

In [32]:
import pandas as pd
import os
import glob

def parse_csv_and_extract_start_times(folder_path, start_row=16, file_pattern="*.csv", encoding='utf-8'):
    """
    Парсит CSV файлы и извлекает времена, соответствующие первому START в столбце Unnamed: 8
    
    Parameters:
    folder_path (str): путь к папке с файлами
    start_row (int): номер строки, с которой начинать парсинг
    file_pattern (str): шаблон для поиска файлов
    encoding (str): кодировка файлов
    
    Returns:
    pd.DataFrame: DataFrame с временами START для каждого файла
    """
    
    # Проверяем существование папки
    if not os.path.exists(folder_path):
        raise ValueError(f"Папка {folder_path} не существует")
    
    # Получаем список CSV файлов
    file_paths = glob.glob(os.path.join(folder_path, file_pattern))
    
    if not file_paths:
        print(f"В папке {folder_path} не найдено файлов по шаблону {file_pattern}")
        return pd.DataFrame()
    
    # Создаем список для хранения результатов
    start_times_data = []
    
    for file_path in file_paths:
        try:
            # Читаем файл, пропуская первые (start_row-1) строк
            df = pd.read_csv(file_path, skiprows=start_row-1, encoding=encoding)
            
            # Получаем имя файла без пути
            file_name = os.path.basename(file_path)
            
            # Проверяем наличие столбца Unnamed: 8
            # if 'Unnamed: 8' not in df.columns:
            #     print(f"В файле {file_name} отсутствует столбец Unnamed: 8")
            #     continue
            
            # Ищем первую строку со значением 'START' в столбце Unnamed: 8
            start_row_data = df[df['Unnamed: 8'] == 'START']
            
            # Берем первую найденную строку со START
            first_start = start_row_data.iloc[-1]
            start_time = first_start['Time']
            
            # Добавляем данные в список
            start_times_data.append({
                'Time': start_time
            })
            
            print(f"Файл: {file_name} - Время START: {start_time}")
            
        except Exception as e:
            print(f"Ошибка при обработке файла {file_path}: {str(e)}")
            start_times_data.append({
                    'Time': '-'
                })
    
    # Создаем DataFrame из результатов
    result_df = pd.DataFrame(start_times_data)
    
    return result_df

def save_start_times_to_csv(result_df, output_file):
    """
    Сохраняет результаты в CSV файл
    
    Parameters:
    result_df (pd.DataFrame): DataFrame с результатами
    output_file (str): путь к выходному файлу
    """
    if not result_df.empty:
        result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"Результаты сохранены в файл: {output_file}")
    else:
        print("Нет данных для сохранения")

starts_7D = parse_csv_and_extract_start_times(folder_path = 'E:/Projects/BOWL/BehaviorData/7_boris_ann', file_pattern="*6D.csv")
save_start_times_to_csv(starts_7D, 'E:/Projects/BOWL/BehaviorData/7_boris_ann/starts_6D.csv')


Файл: BOWL_D01_6D.csv - Время START: 0.533
Файл: BOWL_D03_6D.csv - Время START: 176.7
Ошибка при обработке файла E:/Projects/BOWL/BehaviorData/7_boris_ann\BOWL_D04_6D.csv: 'Unnamed: 8'
Файл: BOWL_D07_6D.csv - Время START: 56.3
Файл: BOWL_D14_6D.csv - Время START: 4.0
Ошибка при обработке файла E:/Projects/BOWL/BehaviorData/7_boris_ann\BOWL_D17_6D.csv: 'Unnamed: 8'
Файл: BOWL_F01_6D.csv - Время START: 51.167
Файл: BOWL_F04_6D.csv - Время START: 214.6
Файл: BOWL_F05_6D.csv - Время START: 299.933
Файл: BOWL_F06_6D.csv - Время START: 299.9
Файл: BOWL_F07_6D.csv - Время START: 71.1
Ошибка при обработке файла E:/Projects/BOWL/BehaviorData/7_boris_ann\BOWL_F08_6D.csv: 'Unnamed: 8'
Файл: BOWL_F09_6D.csv - Время START: 299.867
Файл: BOWL_F11_6D.csv - Время START: 299.9
Файл: BOWL_F14_6D.csv - Время START: 298.333
Файл: BOWL_F15_6D.csv - Время START: 299.932
Файл: BOWL_F26_6D.csv - Время START: 62.133
Файл: BOWL_F28_6D.csv - Время START: 20.367
Файл: BOWL_F29_6D.csv - Время START: 127.433
Файл: 

# для сшивки двух сессий 

In [3]:
import os
import re

def rename_files_simple(directory_path):
    """Упрощенная версия скрипта для переименования файлов"""
    for filename in os.listdir(directory_path):
        old_path = os.path.join(directory_path, filename)
        
        if os.path.isfile(old_path):
            name, ext = os.path.splitext(filename)
            
            # Заменяем "_1_" на "_"
            new_name = name.replace("_1_", "_")
            
            # Прибавляем 24 к числу в конце
            match = re.search(r'(\d+)$', new_name)
            if match:
                number = int(match.group(1)) + 25
                new_name = new_name[:match.start(1)] + str(number)
            
            new_filename = new_name + ext
            new_path = os.path.join(directory_path, new_filename)
            
            if new_filename != filename:
                os.rename(old_path, new_path)
                print(f"'{filename}' -> '{new_filename}'")

# Использование
folder_path = "e:\\Projects\\3DM\\BehaviorData\\1_Raw\\3DM_2wave\\3DM_J30_1_1D\\"  # Замените на ваш путь
rename_files_simple(folder_path)

'3DM_J30_1_1D_0.avi' -> '3DM_J30_1D_25.avi'
'3DM_J30_1_1D_1.avi' -> '3DM_J30_1D_26.avi'
'3DM_J30_1_1D_10.avi' -> '3DM_J30_1D_35.avi'
'3DM_J30_1_1D_11.avi' -> '3DM_J30_1D_36.avi'
'3DM_J30_1_1D_12.avi' -> '3DM_J30_1D_37.avi'
'3DM_J30_1_1D_13.avi' -> '3DM_J30_1D_38.avi'
'3DM_J30_1_1D_14.avi' -> '3DM_J30_1D_39.avi'
'3DM_J30_1_1D_15.avi' -> '3DM_J30_1D_40.avi'
'3DM_J30_1_1D_16.avi' -> '3DM_J30_1D_41.avi'
'3DM_J30_1_1D_17.avi' -> '3DM_J30_1D_42.avi'
'3DM_J30_1_1D_18.avi' -> '3DM_J30_1D_43.avi'
'3DM_J30_1_1D_19.avi' -> '3DM_J30_1D_44.avi'
'3DM_J30_1_1D_2.avi' -> '3DM_J30_1D_27.avi'
'3DM_J30_1_1D_20.avi' -> '3DM_J30_1D_45.avi'
'3DM_J30_1_1D_21.avi' -> '3DM_J30_1D_46.avi'
'3DM_J30_1_1D_22.avi' -> '3DM_J30_1D_47.avi'
'3DM_J30_1_1D_23.avi' -> '3DM_J30_1D_48.avi'
'3DM_J30_1_1D_24.avi' -> '3DM_J30_1D_49.avi'
'3DM_J30_1_1D_25.avi' -> '3DM_J30_1D_50.avi'
'3DM_J30_1_1D_26.avi' -> '3DM_J30_1D_51.avi'
'3DM_J30_1_1D_27.avi' -> '3DM_J30_1D_52.avi'
'3DM_J30_1_1D_28.avi' -> '3DM_J30_1D_53.avi'
'3DM_J30_1_1D

## Копирует по правилу

In [2]:
import os
import re
import shutil
from pathlib import Path

# === НАСТРОЙКИ ПУТЕЙ ===
# Откуда берем данные
source_root = Path(r"i:\Projects\CONS_May-June-25\BehaviorData")

# Куда кладем для Cam1 и Cam2
dest_root_cam1 = Path(r"h:\Projects\CONS\BehaviorData\Cam1_upper\1_Raw")
dest_root_cam2 = Path(r"h:\Projects\CONS\BehaviorData\Cam2_side\1_Raw")

# Паттерн из названия файла
# (чуть поправил [A-z] на [A-Za-z], чтобы не захватывать лишние символы из ASCII)
#pattern = re.compile(r"[A-Za-z0-9]{3,4}_[A-Za-z]{1,2}\d{2}_(\dD|\dT)(_\dT)?")
pattern = re.compile(r"[A-Za-z0-9]{3,4}_[A-Za-z]{1,2}\d{2}_(\d{1,2}[DT])(_\d{1,2}T)?")

print("Старт обхода:", source_root)

for root, dirs, files in os.walk(source_root):
    folder_path = Path(root)
    folder_name = folder_path.name

    # Ищем только папки, в имени которых есть Cam1 или Cam2
    is_cam1 = "Cam1" in folder_name
    is_cam2 = "Cam2" in folder_name

    if not (is_cam1 or is_cam2):
        continue

    # Если внутри есть подпапки — можно предупредить (по условию их быть не должно)
    if dirs:
        print(f"ПРЕДУПРЕЖДЕНИЕ: в папке {folder_path} есть подпапки {dirs}")

    # Если нет файлов – пропускаем
    if not files:
        print(f"Папка {folder_path} пуста, пропускаю")
        continue

    # Собираем паттерны из имен всех файлов в папке
    found_patterns = set()
    for fname in files:
        match = pattern.search(fname)
        if match:
            found_patterns.add(match.group(0))
        else:
            print(f"ПРЕДУПРЕЖДЕНИЕ: файл {fname} в {folder_path} не содержит нужного паттерна")

    if not found_patterns:
        print(f"ВНИМАНИЕ: ни в одном файле из {folder_path} не найден паттерн, папка пропущена")
        continue

    if len(found_patterns) > 1:
        print(f"ВНИМАНИЕ: в папке {folder_path} найдено несколько разных паттернов: {found_patterns}")
        print("Эту папку пропускаю, чтобы не смешивать разные сессии.")
        continue

    # Единственный паттерн для этой папки
    pattern_str = found_patterns.pop()

    # Выбираем корень назначения в зависимости от типа камеры
    if is_cam1:
        dest_root = dest_root_cam1
    else:
        dest_root = dest_root_cam2

    # Итоговая папка, куда копируем файлы
    dest_folder = dest_root / pattern_str
    dest_folder.mkdir(parents=True, exist_ok=True)

    # Копирование всех файлов из исходной папки
    copied = 0
    for fname in files:
        src = folder_path / fname
        if not src.is_file():
            continue
        dst = dest_folder / fname
        shutil.copy2(src, dst)
        copied += 1

    print(f"Скопировано {copied} файлов из {folder_path} в {dest_folder}")


Старт обхода: i:\Projects\CONS_May-June-25\BehaviorData
Скопировано 38 файлов из i:\Projects\CONS_May-June-25\BehaviorData\FM_CONS_1D_10T\CONS_F06_1D_10T\CONS_Cam1_10T в h:\Projects\CONS\BehaviorData\Cam1_upper\1_Raw\CONS_F06_1D_10T
Скопировано 37 файлов из i:\Projects\CONS_May-June-25\BehaviorData\FM_CONS_1D_10T\CONS_F06_1D_10T\CONS_Cam2_10T в h:\Projects\CONS\BehaviorData\Cam2_side\1_Raw\CONS_F06_1D_10T
Скопировано 38 файлов из i:\Projects\CONS_May-June-25\BehaviorData\FM_CONS_1D_10T\CONS_F26_1D_10T\CONS_Cam1_10T в h:\Projects\CONS\BehaviorData\Cam1_upper\1_Raw\CONS_F26_1D_10T
Скопировано 37 файлов из i:\Projects\CONS_May-June-25\BehaviorData\FM_CONS_1D_10T\CONS_F26_1D_10T\CONS_Cam2_10T в h:\Projects\CONS\BehaviorData\Cam2_side\1_Raw\CONS_F26_1D_10T
Скопировано 38 файлов из i:\Projects\CONS_May-June-25\BehaviorData\FM_CONS_1D_10T\CONS_F54_1D_10T\CONS_Cam1_10T в h:\Projects\CONS\BehaviorData\Cam1_upper\1_Raw\CONS_F54_1D_10T
Скопировано 37 файлов из i:\Projects\CONS_May-June-25\Behavio

In [6]:
import re
from pathlib import Path

# Папка с файлами
folder = Path(r"h:\Projects\CONS\CalciumData\1_Raw\CONS_F01_1D_1T")

# Только посмотреть изменения, не применять
dry_run = False

# Паттерн для поиска номера перед T
pattern = re.compile(r"^(?P<prefix>.*?_)(?P<num>\d+)(?P<suffix>T_.*)$")

# --- Функция натуральной сортировки ---
def natural_key(s):
    # Разбиваем строку на числовые и текстовые части
    return [int(text) if text.isdigit() else text for text in re.split(r"(\d+)", s.name)]

# Получаем файлы и сортируем в натуральном порядке, затем реверсируем
files = sorted([f for f in folder.iterdir() if f.is_file()], key=natural_key, reverse=True)

for file in files:
    name = file.name
    m = pattern.match(name)

    if not m:
        print(f"Пропускаю (не подходит под шаблон): {name}")
        continue

    num = int(m.group("num"))
    if num < 6:
        print(f"Оставляю (num < 6): {name}")
        continue

    new_num = num + 1
    new_name = f"{m.group('prefix')}{new_num}{m.group('suffix')}"
    new_path = file.with_name(new_name)

    print(f"{name}  -->  {new_name}")

    if not dry_run:
        file.rename(new_path)


CONS_F01_1D_8T_15.avi  -->  CONS_F01_1D_9T_15.avi
CONS_F01_1D_8T_14.avi  -->  CONS_F01_1D_9T_14.avi
CONS_F01_1D_8T_13.avi  -->  CONS_F01_1D_9T_13.avi
CONS_F01_1D_8T_12.avi  -->  CONS_F01_1D_9T_12.avi
CONS_F01_1D_8T_11.avi  -->  CONS_F01_1D_9T_11.avi
CONS_F01_1D_8T_10.avi  -->  CONS_F01_1D_9T_10.avi
CONS_F01_1D_8T_9.avi  -->  CONS_F01_1D_9T_9.avi
CONS_F01_1D_8T_8.avi  -->  CONS_F01_1D_9T_8.avi
CONS_F01_1D_8T_7.avi  -->  CONS_F01_1D_9T_7.avi
CONS_F01_1D_8T_6.avi  -->  CONS_F01_1D_9T_6.avi
CONS_F01_1D_8T_5.avi  -->  CONS_F01_1D_9T_5.avi
CONS_F01_1D_8T_4.avi  -->  CONS_F01_1D_9T_4.avi
CONS_F01_1D_8T_3.avi  -->  CONS_F01_1D_9T_3.avi
CONS_F01_1D_8T_2.avi  -->  CONS_F01_1D_9T_2.avi
CONS_F01_1D_8T_1.avi  -->  CONS_F01_1D_9T_1.avi
CONS_F01_1D_8T_0.avi  -->  CONS_F01_1D_9T_0.avi
CONS_F01_1D_7T_35.avi  -->  CONS_F01_1D_8T_35.avi
CONS_F01_1D_7T_34.avi  -->  CONS_F01_1D_8T_34.avi
CONS_F01_1D_7T_33.avi  -->  CONS_F01_1D_8T_33.avi
CONS_F01_1D_7T_32.avi  -->  CONS_F01_1D_8T_32.avi
CONS_F01_1D_7T_31.av

In [8]:
import re
from pathlib import Path

# Папка, где нужно откатить переименование
folder = Path(r"h:\Projects\CONS\CalciumData\1_Raw\CONS_F06_1D_1T")

# Режим предпросмотра: True = только показать, False = реально переименовать
dry_run = False

# Паттерн: prefix_ + ЧИСЛО + "T_..."
# Примеры:
#   CONS_F06_1D_7T_1.avi
#   что-угодно_10T_2.avi
pattern = re.compile(r"^(?P<prefix>.*?_)(?P<num>\d+)(?P<suffix>T_.*)$")

def natural_key(p: Path):
    """Ключ для натуральной сортировки по имени файла."""
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", p.name)]

# Берём только файлы, сортируем по натуральному порядку (1,2,...,9,10,...)
files = sorted([f for f in folder.iterdir() if f.is_file()], key=natural_key)

for file in files:
    name = file.name
    m = pattern.match(name)

    if not m:
        print(f"Пропускаю (не подходит под шаблон): {name}")
        continue

    num = int(m.group("num"))

    # Откатываем только то, что потенциально уже было увеличено (7, 8, 9, 10, ...)
    if num < 7:
        print(f"Оставляю (num < 7): {name}")
        continue

    new_num = num - 1
    new_name = f"{m.group('prefix')}{new_num}{m.group('suffix')}"
    new_path = file.with_name(new_name)

    if new_path.exists():
        print(f"ВНИМАНИЕ: целевой файл уже существует, пропускаю: {name} -> {new_name}")
        continue

    print(f"{name}  -->  {new_name}")

    if not dry_run:
        file.rename(new_path)


Оставляю (num < 7): CONS_F06_1D_1T_0.avi
Оставляю (num < 7): CONS_F06_1D_1T_1.avi
Оставляю (num < 7): CONS_F06_1D_1T_2.avi
Оставляю (num < 7): CONS_F06_1D_1T_3.avi
Оставляю (num < 7): CONS_F06_1D_1T_4.avi
Оставляю (num < 7): CONS_F06_1D_1T_5.avi
Оставляю (num < 7): CONS_F06_1D_1T_6.avi
Оставляю (num < 7): CONS_F06_1D_1T_7.avi
Оставляю (num < 7): CONS_F06_1D_1T_8.avi
Оставляю (num < 7): CONS_F06_1D_1T_Mini_TS.csv
Оставляю (num < 7): CONS_F06_1D_2T_0.avi
Оставляю (num < 7): CONS_F06_1D_2T_1.avi
Оставляю (num < 7): CONS_F06_1D_2T_2.avi
Оставляю (num < 7): CONS_F06_1D_2T_3.avi
Оставляю (num < 7): CONS_F06_1D_2T_4.avi
Оставляю (num < 7): CONS_F06_1D_2T_5.avi
Оставляю (num < 7): CONS_F06_1D_2T_6.avi
Оставляю (num < 7): CONS_F06_1D_2T_7.avi
Оставляю (num < 7): CONS_F06_1D_2T_8.avi
Оставляю (num < 7): CONS_F06_1D_2T_9.avi
Оставляю (num < 7): CONS_F06_1D_2T_10.avi
Оставляю (num < 7): CONS_F06_1D_2T_11.avi
Оставляю (num < 7): CONS_F06_1D_2T_12.avi
Оставляю (num < 7): CONS_F06_1D_2T_13.avi
Оставл